In [1]:
!pip install -U gradio groq python-dotenv

In [2]:
import gradio as gr
from groq import Groq
import json

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.3-70b-versatile"

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role":"user",
            "content":"Say Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [5]:
def create_prompt(domain, interest, skills, budget, audience, country):

    return f"""
You are an expert startup consultant.

Generate one startup idea.

Return ONLY JSON.

Format:

{{
"overview":"",
"validation":"",
"revenue":"",
"competitors":"",
"marketing":""
}}

Domain : {domain}

Interest : {interest}

Skills : {skills}

Budget : {budget}

Audience : {audience}

Country : {country}
"""

In [6]:
def generate_business_idea(domain,
                           interest,
                           skills,
                           budget,
                           audience,
                           country):

    prompt = create_prompt(
        domain,
        interest,
        skills,
        budget,
        audience,
        country
    )

    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]

    )

    result = response.choices[0].message.content

    try:

        data = json.loads(result)

        return (
            data["overview"],
            data["validation"],
            data["revenue"],
            data["competitors"],
            data["marketing"]
        )

    except Exception as e:

        return (
            result,
            "",
            "",
            "",
            ""
        )

In [7]:
import gradio as gr


with gr.Blocks() as demo:

    gr.Markdown("# 🚀 AI Business Idea Generator")

    with gr.Row():

        domain = gr.Textbox(
            label="Business Domain"
        )

        interest = gr.Textbox(
            label="Your Interest"
        )


    skills = gr.Textbox(
        label="Skills"
    )

    budget = gr.Textbox(
        label="Budget"
    )

    audience = gr.Textbox(
        label="Target Audience"
    )

    country = gr.Textbox(
        label="Country"
    )


    generate_btn = gr.Button(
        "Generate Idea 🚀"
    )

    clear_btn = gr.Button(
        "Clear"
    )


    # OUTPUTS

    overview = gr.Markdown()
    validation = gr.Markdown()
    revenue = gr.Markdown()
    competitors = gr.Markdown()
    marketing = gr.Markdown()



    # ==========================
    # BUTTON EVENTS
    # ==========================

    generate_btn.click(
        fn=generate_business_idea,
        inputs=[
            domain,
            interest,
            skills,
            budget,
            audience,
            country
        ],
        outputs=[
            overview,
            validation,
            revenue,
            competitors,
            marketing
        ]
    )


    clear_btn.click(
        fn=lambda: (
            "",
            "",
            "",
            "",
            "",
        ),
        outputs=[
            overview,
            validation,
            revenue,
            competitors,
            marketing
        ]
    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
